# Run Pipeline

Executes the full analysis pipeline in order by running each notebook non-interactively.
No changes to the other notebooks are needed — this just triggers them externally.

**Pipeline order:**
1. `00_prepare_data_nwb.ipynb` — extract NWB intersections → `intersections.gpkg`
2. `02_image_extraction.ipynb` — filter 706k photos to ~170k near intersections
3. `04_leg_selection.ipynb` — match one photo per leg, compute u_deg
4. `05_reprojection.ipynb` — reproject photos centered on intersection per leg

**Not included** (run manually when needed):
- `util_prepare_data_osm.ipynb` — OSM-based alternative to 00, kept for reference
- `util_image_sampling.ipynb` — one-off sample copy, not part of main pipeline
- `exp_image_cropping_comparison.ipynb` — exploratory comparison, not part of main pipeline

**Stopping behaviour:** if any notebook fails, the pipeline stops immediately and prints the error.
Fix the issue in the failing notebook, then re-run from that step.

**Testing mode:** notebooks 04 and 05 have `N_INTERSECTIONS = 50` by default.
Set that to `None` in those notebooks before running for the full dataset.

In [ ]:
import subprocess
import sys
import os
import time

# All notebook paths are relative to this folder
NOTEBOOKS_DIR = os.path.dirname(os.path.abspath('__file__'))

# Pipeline steps — comment out any step you want to skip
# Format: (filename, description)
PIPELINE = [
    ("00_prepare_data_nwb.ipynb",  "Prepare NWB intersections"),
    ("02_image_extraction.ipynb",  "Filter photos near intersections"),
    ("04_leg_selection.ipynb",     "Match photos to intersection legs"),
    ("05_reprojection.ipynb",      "Reproject images per leg"),
]

In [ ]:
def run_notebook(filename):
    """Execute a single notebook in-place using nbconvert.
    
    --execute        : runs all cells top to bottom
    --inplace        : saves output back into the same .ipynb file
    --ExecutePreprocessor.timeout : seconds before a single cell times out
                       set high because image processing cells can take a long time
    """
    return subprocess.run(
        [
            "jupyter", "nbconvert",
            "--to", "notebook",
            "--execute",
            "--inplace",
            "--ExecutePreprocessor.timeout=7200",  # 2 hour timeout per cell
            filename
        ],
        capture_output=True,
        text=True,
        cwd=NOTEBOOKS_DIR  # run from the notebooks folder so relative paths work
    )


# Run each step in order
pipeline_start = time.time()

for filename, description in PIPELINE:
    print(f"{'='*60}")
    print(f"Step: {description}")
    print(f"File: {filename}")
    print(f"{'='*60}")

    step_start = time.time()
    result = run_notebook(filename)
    elapsed = time.time() - step_start

    if result.returncode != 0:
        # Print the error and stop — don't run later steps on broken output
        print(f"FAILED after {elapsed:.0f}s")
        print()
        print("--- stderr ---")
        print(result.stderr)
        print()
        print("Fix the issue in the notebook above, then re-run from that step.")
        sys.exit(1)
    else:
        print(f"Done in {elapsed:.0f}s")
        print()

total = time.time() - pipeline_start
print(f"{'='*60}")
print(f"Pipeline complete. Total time: {total/60:.1f} minutes")
print(f"{'='*60}")